In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

movies = pd.read_csv("../data/ml-latest-small/movies.csv")
tags = pd.read_csv("../data/ml-latest-small/tags.csv")

print(movies.shape, tags.shape)

(9742, 3) (3683, 4)


In [2]:
tags_por_filme = (
    tags.assign(tag=tags["tag"].astype(str).str.lower())
    .groupby("movieId")["tag"]
    .apply(" ".join)
    .reset_index(name="tags")
)

movies = movies.merge(tags_por_filme, on="movieId", how="left")
movies["tags"] = movies["tags"].fillna("")

movies[["title", "genres", "tags"]].head()

,title,genres,tags
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar pixar fun
1,Jumanji (1995),Adventure|Children|Fantasy,fantasy magic board game robin williams game
2,Grumpier Old Men (1995),Comedy|Romance,moldy old
3,Waiting to Exhale (1995),Comedy|Drama|Romance,
4,Father of the Bride Part II (1995),Comedy,pregnancy remake


In [3]:
movies["genres_limpo"] = (
    movies["genres"]
    .replace("(no genres listed)", "")
    .str.replace("|", " ", regex=False)
    .str.lower()
)

movies["texto"] = movies["genres_limpo"] + " " + movies["tags"]
movies["texto"].head()

0    adventure animation children comedy fantasy pi...
1    adventure children fantasy fantasy magic board...
2                             comedy romance moldy old
3                                comedy drama romance 
4                              comedy pregnancy remake
Name: texto, dtype: str

In [4]:
tfidf = TfidfVectorizer(stop_words="english")
matriz = tfidf.fit_transform(movies["texto"])

print("Formato da matriz:", matriz.shape)

Formato da matriz: (9742, 1675)


In [5]:
indices = pd.Series(movies.index, index=movies["movieId"])


def buscar_id(trecho):
    """Procura o movieId pelo título."""
    achados = movies[movies["title"].str.contains(trecho, case=False, regex=False)]
    return achados[["movieId", "title"]]


def similares(movie_id, n=10):
    """Retorna os n filmes mais parecidos com o filme informado."""
    idx = indices[movie_id]
    scores = cosine_similarity(matriz[idx], matriz).flatten()
    top = scores.argsort()[::-1][1 : n + 1]  # pula o próprio filme

    resultado = movies.iloc[top][["movieId", "title", "genres"]].copy()
    resultado["similaridade"] = scores[top].round(3)
    return resultado

In [6]:
buscar_id("Toy Story")

,movieId,title
0,1,Toy Story (1995)
2355,3114,Toy Story 2 (1999)
7355,78499,Toy Story 3 (2010)


In [7]:
similares(1)

,movieId,title,genres,similaridade
1757,2355,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,0.862
2355,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.644
8695,122918,Guardians of the Galaxy 2 (2017),Action|Adventure|Sci-Fi,0.368
8219,103755,Turbo (2013),Adventure|Animation|Children|Comedy|Fantasy,0.358
8927,136016,The Good Dinosaur (2015),Adventure|Animation|Children|Comedy|Fantasy,0.358
3000,4016,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,0.358
9430,166461,Moana (2016),Adventure|Animation|Children|Comedy|Fantasy,0.358
6486,53121,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy,0.358
6194,45074,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy,0.358
6948,65577,"Tale of Despereaux, The (2008)",Adventure|Animation|Children|Comedy|Fantasy,0.358


In [1]:
import pandas as pd

df = pd.read_csv("../data/movies_enriched.csv")
print(df.shape)
print("Filmes com sinopse:", (df["overview"].fillna("") != "").mean().round(3))
df[["title", "director", "cast", "poster_path"]].head(10)

(9742, 16)
Filmes com sinopse: 0.987


,title,director,cast,poster_path
0,Toy Story (1995),John Lasseter,Tom Hanks|Tim Allen|Don Rickles|Jim Varney|Wal...,/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg
1,Jumanji (1995),Joe Johnston,Robin Williams|Kirsten Dunst|Bradley Pierce|Bo...,/bdHG5Mo83VPobeZZdlSz0Y7HQHB.jpg
2,Grumpier Old Men (1995),Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret|Sophia ...,/1FSXpj5e8l4KH6nVFO5SPUeraOt.jpg
3,Waiting to Exhale (1995),Forest Whitaker,Whitney Houston|Angela Bassett|Loretta Devine|...,/4wjGMwPsdlvi025ZqR4rXnFDvBz.jpg
4,Father of the Bride Part II (1995),Charles Shyer,Steve Martin|Diane Keaton|Martin Short|Kimberl...,/rj4LBtwQ0uGrpBnCELr716Qo3mw.jpg
5,Heat (1995),Michael Mann,Al Pacino|Robert De Niro|Val Kilmer|Jon Voight...,/umSVjVdbVwtx5ryCA2QXL44Durm.jpg
6,Sabrina (1995),Sydney Pollack,Harrison Ford|Julia Ormond|Greg Kinnear|Nancy ...,/i8PbLJDPU7vCwwscWD625oHbJy.jpg
7,Tom and Huck (1995),Peter Hewitt,Jonathan Taylor Thomas|Brad Renfro|Eric Schwei...,/bMY31ikEOIPOHqQwhPhII8UVBln.jpg
8,Sudden Death (1995),Peter Hyams,Jean-Claude Van Damme|Powers Boothe|Raymond J....,/4Snihptli0we9I4W5QfufOdHSeV.jpg
9,GoldenEye (1995),Martin Campbell,Pierce Brosnan|Sean Bean|Izabella Scorupco|Fam...,/z0ljRnNxIO7CRBhLEO0DvLgAFPR.jpg
